# Multi-anchor extension: nearest-anchor heuristic vs. a univariate KF + RTS smoother

[`ssp_extend_prior.ipynb`](./ssp_extend_prior.ipynb) walks through `extend_states_prior_nearest`, which fills
every undisclosed step of an anchored state with the **nearest**-anchor random-walk marginal,

$$a_{\text{obs}}[t] = a^{*}, \qquad P_{\text{obs}}[t] = P^{*} + |t - t^{*}|\,Q.$$

Its own docstring flags a limit: this is the exact marginal only for an **isolated channel with a
single anchor**. This notebook checks that claim for a channel with **several** anchors, and builds
out the principled alternative.

**The method under test (the proposed approach).** Since the state priors are not diffused across
states here — each disclosed state is its own input, coupled only later in the augmented-measurement
step — treat each state column as an *independent univariate time series* and run a standard
linear-Gaussian smoother over the same driftless random walk $x_t = x_{t-1} + w_t$,
$\mathrm{Var}(w_t) = Q$:

1. **Forward Kalman filter.** A disclosed anchor is fed in as an observation $(a^{*}, P^{*})$; an
   undisclosed step is a *missing observation*, so the filter just predicts — carry the mean, grow
   the variance by $Q$. The filter handles the `inf`/null steps natively; there's nothing special to
   do.
2. **Backward RTS smoother.** A second pass propagates each anchor's information *backward* too, so
   every step sees anchors on both sides.

For this linear-Gaussian model the RTS smoother is **exact** — it returns the true per-step posterior
marginal, no simulation smoother required. (A simulation smoother only earns its keep if we later
carry the nonlinear positivity reparam $\lambda = e^{k a}$ through the extension, or want *joint*
draws across time rather than per-step marginals — see the closing note.)

Unlike the nearest-anchor rule, this fuses *every* anchor in the channel through the shared Markov
chain — including, as we'll see, revising the anchors themselves.

In [ ]:
import jax

jax.config.update("jax_enable_x64", True)  # float64 for a clean diffuse-prior smoother

import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from bunobee.models.ssp.plotting import plot_states
from bunobee.models.ssp.prior import (
    disclosed_idx,
    extend_states_prior_nearest,
    extend_states_prior_smoothed,
)


## 1. A channel with several disclosed anchors

Two states, hand-built so each anchor can carry its own mean and variance (unlike
`construct_states_prior`, which discloses one constant value per window). The disclosed values are
elasticity-like coefficients, so they sit in `[0, 1]`:

- `single_anchor` -- one anchor at `t=20`. This is the case `extend_states_prior_nearest` is exact for; it's
  the control.
- `multi_anchor` -- three anchors at `t=5, 20, 34` with different means and precisions. The middle
  anchor (`t=20`) is deliberately the **loose outlier** -- a somewhat higher value with large
  variance -- so the two tighter neighbours should pull it back down once the channel is smoothed.
  This is the case under test.

The disclosed values live in an editable `anchor_spec` at the top of the next cell -- tweak the
means / variances there and re-run to reconfigure the demo. (Changing the means/variances is safe;
changing the timesteps or the set of states also means updating the hardcoded `rows` / `cross_t` /
`s` picks in the later cells.)

In [ ]:
n_steps = 40
dates = pd.date_range("2024-01-07", periods=n_steps, freq="W")

# ---------------------------------------------------------------------------
# Anchor spec -- EDIT HERE to reconfigure the demo. One row per disclosed step:
#   (time_index, mean, variance)
#     mean     : elasticity-like coefficient, kept in [0, 1]
#     variance : disclosure tightness -- small = tight/confident, large = loose
# The `multi_anchor` middle row is the loose outlier (somewhat higher mean + large
# variance), so its two tight neighbours pull it back down once the channel is smoothed.
# ---------------------------------------------------------------------------
anchor_spec = {
    "single_anchor": [(20, 0.60, 0.03)],
    "multi_anchor": [(5, 0.30, 0.02), (20, 0.65, 0.15), (34, 0.35, 0.02)],
}

state_labels = list(anchor_spec)
n_states = len(state_labels)

a_obs = np.zeros((n_steps, n_states))
p_obs = np.full((n_steps, n_states), np.inf)
for s, label in enumerate(state_labels):
    for t, a_star, p_star in anchor_spec[label]:
        a_obs[t, s], p_obs[t, s] = a_star, p_star

base_prior = xr.Dataset(
    {
        "a_obs": (("time", "state"), a_obs),
        "P_obs": (("time", "state"), p_obs),
        "positivity": (("state",), np.zeros(n_states, dtype=bool)),
    },
    coords={"time": np.arange(n_steps), "state": state_labels},
)

anchors = disclosed_idx(base_prior)
print("disclosed steps:", anchors)
base_prior[["a_obs", "P_obs"]].isel(time=anchors).to_dataframe()

## 2. The univariate KF + RTS smoother, packaged as `extend_states_prior_smoothed`

The exact alternative ships in the package as
[`extend_states_prior_smoothed`](../../src/bunobee/models/ssp/prior.py) -- it reuses bunobee's own
diagonal 1-D Kalman filter and RTS smoother from `bunobee.models.ssp.kalman_1d`, the same code the
SSP models run in production:

- `kalman_filter_1d(...)` -- forward pass. Its optional **state-fusion** step precision-merges each
  disclosed anchor $(a^{*}, P^{*})$ into the carried state; steps left at `P_obs = inf` are no-ops
  (pure predict-through). That is exactly the *soft-observation* reading below.
- `kalman_rts_smoother_1d(at, Pt, sigma_q)` -- backward pass, returning smoothed means **and**
  marginal variances. It conditions on the fused anchors implicitly (they are already baked into
  `at`, `Pt`), so no extra arguments are needed.

**Driving it in "extension mode."** The filter is built around a scalar observation
$y_t = Z_t' \alpha_t + \varepsilon_t$ that couples the states through the design row $Z_t$. Here we
have no such $y$ -- only the disclosed anchors -- so `extend_states_prior_smoothed` passes a **zero
design matrix** $Z = 0$. That zeroes the Kalman gain, so the $y$-update contributes nothing and every
state evolves as an *independent* univariate random walk fed only by its own anchors. Two conventions
are handled internally: the filter takes a process-noise **standard deviation**, so it passes
`sigma_q = sqrt(Q)`; and the diffuse initial variance is a large finite `P0_diffuse` (default `1e8`,
comfortable in float64) standing in for "no prior information."

**Why fusion, not injection.** The anchor enters as a *soft observation* -- a measurement with noise
variance $R = P^{*}$ -- not a hard pin. When the incoming prediction is still diffuse (first anchor
reached) the posterior collapses to $(a^{*}, P^{*})$ exactly, so an isolated anchor behaves like the
heuristic; but when information is already flowing in from another anchor, the two are *fused* by
inverse-variance weighting. That fusion -- $P^{*}$ read as a stated *uncertainty* rather than a hard
value -- is the whole reason the multi-anchor result departs from the nearest-anchor rule.

(One consequence of the diffuse `P0`: inside the filter a state with *no* anchor comes back
diffuse-but-finite; `extend_states_prior_smoothed` resets those columns to `inf`, matching
`extend_states_prior_nearest`. Both states below are anchored, so it doesn't arise.)


## 3. Heuristic vs. smoother, side by side

Run both extensions on the same base prior and line up a handful of representative steps: each
anchor, the midpoints between anchors, and a couple of steps outside the anchor span. The `reference`
columns below are the KF + RTS smoother from section 2.

In [ ]:
Q = 0.02  # per-state process variance sigma_q**2, same default as ssp_extend_prior.ipynb

heuristic = extend_states_prior_nearest(base_prior, Q)
reference = extend_states_prior_smoothed(base_prior, Q)  # exact KF + RTS marginal

rows = [0, 5, 12, 20, 27, 34, 39]  # anchors (5, 20, 34), their midpoints, and both tails
comparison = pd.DataFrame(
    {
        "a_heuristic": heuristic["a_obs"].values[rows, 1],
        "P_heuristic": heuristic["P_obs"].values[rows, 1],
        "a_reference": reference["a_obs"].values[rows, 1],
        "P_reference": reference["P_obs"].values[rows, 1],
    },
    index=pd.Index(rows, name="t"),
)
comparison["d_mean"] = comparison["a_reference"] - comparison["a_heuristic"]
comparison["P_ratio (heuristic / reference)"] = comparison["P_heuristic"] / comparison["P_reference"]
comparison.round(4)


Even at `t=20` -- an anchor itself -- the reference revises the disclosed value: because it treats
`P*` as genuine observation noise rather than a hard pin, the tighter anchors at `t=5` and `t=34` pull
the smoothed estimate away from the raw disclosure. `extend_states_prior_nearest` freezes every anchor exactly
(by construction, see `test_anchor_entry_preserved_exactly` in
`tests/test_extend_states_prior_nearest.py`), so it cannot reproduce that. Between anchors, the reference's
`P_ratio` is consistently `>= 1`: fusing two directions of information is never less informative than
picking the single nearer anchor.

In [ ]:
fig, axes = plot_states(
    {
        "anchors only": np.broadcast_to(base_prior["a_obs"].values, (1, n_steps, n_states)),
        "nearest-anchor heuristic": np.broadcast_to(heuristic["a_obs"].values, (1, n_steps, n_states)),
        "in-house KF + RTS smoother": np.broadcast_to(reference["a_obs"].values, (1, n_steps, n_states)),
    },
    dates.values,
    state_labels,
    states_key=["anchors only", "nearest-anchor heuristic", "in-house KF + RTS smoother"],
    obs_idx=disclosed_idx(base_prior),
    a_obs=base_prior["a_obs"].values,
    P_obs=base_prior["P_obs"].values,
    title=f"Nearest-anchor heuristic vs. in-house KF + RTS smoother (Q={Q})",
    n_cols=2,
    colors={
        "anchors only": "darkgreen",
        "nearest-anchor heuristic": "steelblue",
        "in-house KF + RTS smoother": "darkorange",
    },
)
plt.show()

`plot_states` draws credible-interval ribbons from Monte Carlo draws; with only one "draw" per method
here (we want the exact `(a_obs, P_obs)` curve, not a resampled one) the 90% ribbon collapses onto
`+/- 1.645 sigma` around that curve, which is exactly what we want to compare. Look at the
`multi_anchor` panel: the heuristic ribbon has a visible **kink** where the nearest anchor switches
(around the midpoint between two anchors), and its cone is wider than the reference's smooth,
narrower one. The `single_anchor` panel should show the two methods overlapping almost exactly.

## 4. Single anchor still matches exactly; multi-anchor does not

A direct numeric gut-check across the whole series, not just the sampled rows above.

In [ ]:
for i, label in enumerate(state_labels):
    d_mean = np.abs(reference["a_obs"].values[:, i] - heuristic["a_obs"].values[:, i]).max()
    d_var = np.abs(reference["P_obs"].values[:, i] - heuristic["P_obs"].values[:, i]).max()
    print(f"{label:>14}: max|delta mean| = {d_mean:.2e}   max|delta variance| = {d_var:.2e}")

## 5. The final smoothed distribution

The two-pass smoother returns a full Gaussian marginal $\mathcal{N}(a_{\text{obs}}[t],
P_{\text{obs}}[t])$ at every step -- that *is* the final distribution the extension produces. Two
views below:

- **Density field** -- the smoother's posterior density across the whole horizon (bright = high
  density). The mean threads through the anchors and the band pinches to each disclosure, widening
  between them.
- **Cross-sections** -- the same distribution sliced at three timesteps as actual bell curves, with
  the nearest-anchor heuristic overlaid so the mean shift and variance tightening read as densities.

In [ ]:
def gaussian_pdf(y: np.ndarray, mean: float, var: float) -> np.ndarray:
    """Evaluate the 1-D Gaussian density N(y; mean, var) pointwise over `y`.

    The atomic building block of the density plots. It turns one per-step marginal ``(mean, var)``
    -- a single column of the smoother output -- into a probability *density* over candidate state
    values ``y``::

        p(y) = exp(-(y - mean) ** 2 / (2 * var)) / sqrt(2 * pi * var)

    The arithmetic is elementwise, so passing a whole grid of ``y`` returns the entire bell curve in
    one call. Both the density-field heatmap (through :func:`density_field`) and the cross-section
    bells call this; it is the only step that actually converts moments into a plottable density.

    Parameters
    ----------
    y : np.ndarray
        State value(s) at which to evaluate the density; any shape, applied elementwise.
    mean : float
        Marginal mean ``a_obs[t]`` of the step being drawn.
    var : float
        Marginal variance ``P_obs[t]`` of the step being drawn; must be finite and positive (every
        step is finite once the smoother has run).

    Returns
    -------
    np.ndarray
        Density values, same shape as ``y``.
    """
    return np.exp(-0.5 * (y - mean) ** 2 / var) / np.sqrt(2 * np.pi * var)


def density_field(a: np.ndarray, P: np.ndarray, n_grid: int = 400, pad: float = 3.5):
    """Assemble the smoother's per-step marginals into a 2-D density field for a heatmap.

    This is the end-to-end density calculation for one state, in three moves:

    1. **Moments (upstream).** :func:`extend_states_prior_smoothed` has already reduced the channel to a
       per-step mean/variance ``(a[t], P[t])``; those arrays are this function's inputs.
    2. **Shared value grid.** Build one vertical axis ``y`` of candidate state values covering every
       step's bell at once, spanning ``[min_t(a - pad*sd), max_t(a + pad*sd)]`` with ``sd = sqrt(P)``.
       A single common grid is what makes the result a rectangular image.
    3. **Vectorised evaluation.** Broadcast the grid as a column (``y[:, None]``) against the moments
       as rows (``a[None, :]``, ``P[None, :]``) through :func:`gaussian_pdf`, filling every
       ``(value, timestep)`` cell in one call. Column ``t`` is exactly the bell ``N(., a[t], P[t])``.

    The returned block drops straight into ``imshow`` (rows = value, columns = time): brighter cells
    are higher density and the bright ridge traces the smoothed mean.

    Parameters
    ----------
    a, P : np.ndarray, shape (n_steps,)
        Per-step marginal mean and variance for one state, as returned by
        :func:`extend_states_prior_smoothed`. ``P`` must be finite (true post-smoother).
    n_grid : int, optional
        Number of points on the shared value grid, by default 400. Higher is smoother but uses more
        memory (the density block is ``n_grid x n_steps``).
    pad : float, optional
        Grid half-width in standard deviations beyond the mean envelope, by default 3.5 (captures
        ~99.95% of every bell's mass).

    Returns
    -------
    y : np.ndarray, shape (n_grid,)
        The shared value grid -- the heatmap's vertical axis.
    density : np.ndarray, shape (n_grid, n_steps)
        Gaussian density at each ``(value, timestep)``; each column integrates to ~1 over ``y``.
    """
    sd = np.sqrt(P)
    y = np.linspace((a - pad * sd).min(), (a + pad * sd).max(), n_grid)
    return y, gaussian_pdf(y[:, None], a[None, :], P[None, :])


anch = disclosed_idx(base_prior)
fig, axes = plt.subplots(1, n_states, figsize=(14, 4))
for i, (ax, label) in enumerate(zip(axes, state_labels)):
    a_i, P_i = reference["a_obs"].values[:, i], reference["P_obs"].values[:, i]
    y, dens = density_field(a_i, P_i)
    im = ax.imshow(
        dens, origin="lower", aspect="auto", extent=[0, n_steps - 1, y[0], y[-1]], cmap="magma"
    )
    ax.plot(np.arange(n_steps), a_i, color="white", lw=1.0, label="smoothed mean")
    m = np.isfinite(base_prior["P_obs"].values[anch, i])
    ax.scatter(
        anch[m], base_prior["a_obs"].values[anch[m], i],
        s=30, marker="x", color="cyan", zorder=3, label="raw anchor",
    )
    ax.set_title(f"{label}: final smoothed density", fontsize=9)
    ax.set_xlabel("time step")
    ax.legend(fontsize=7, loc="upper right")
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04, label="density")
axes[0].set_ylabel("state value")
plt.tight_layout()
plt.show()

In [ ]:
s = 1  # multi_anchor
cross_t = [5, 12, 20]  # tight anchor, midpoint between two anchors, loose anchor
fig, axes = plt.subplots(1, len(cross_t), figsize=(14, 3.6))
for ax, t in zip(axes, cross_t):
    a_h, P_h = heuristic["a_obs"].values[t, s], heuristic["P_obs"].values[t, s]
    a_r, P_r = reference["a_obs"].values[t, s], reference["P_obs"].values[t, s]
    lo = min(a_h - 3.5 * np.sqrt(P_h), a_r - 3.5 * np.sqrt(P_r))
    hi = max(a_h + 3.5 * np.sqrt(P_h), a_r + 3.5 * np.sqrt(P_r))
    y = np.linspace(lo, hi, 400)
    for a_m, P_m, c, name in [
        (a_h, P_h, "steelblue", "nearest-anchor heuristic"),
        (a_r, P_r, "darkorange", "in-house smoother (final)"),
    ]:
        pdf = gaussian_pdf(y, a_m, P_m)
        ax.plot(y, pdf, color=c, label=name)
        ax.fill_between(y, pdf, color=c, alpha=0.15)
    if np.isfinite(base_prior["P_obs"].values[t, s]):
        ax.axvline(base_prior["a_obs"].values[t, s], color="crimson", ls=":", lw=1.0, label="raw anchor mean")
    ax.set_title(f"multi_anchor @ t={t}", fontsize=9)
    ax.set_xlabel("state value")
axes[0].set_ylabel("density")
axes[0].legend(fontsize=7)
plt.tight_layout()
plt.show()

The `t=20` panel is the headline: the raw disclosure (crimson) sits above its neighbours at `0.65`,
but it was disclosed *loosely* (large `P*`), so while the heuristic (blue) reproduces it as a wide
bell, the in-house smoother (orange) is a narrower bell pulled down to about `0.49` -- toward the
tight neighbours at `t=5` (`0.30`) and `t=34` (`0.35`). At `t=12` (a midpoint) the smoother is
visibly tighter than the heuristic; at `t=5` (a tight anchor) they nearly coincide -- the
single-anchor limit in miniature.

## Takeaways

- **Single anchor per channel:** the nearest-anchor heuristic *is* the exact KF + RTS smoother
  marginal -- `extend_states_prior_nearest`'s own docstring caveat ("exact marginal only for an isolated
  channel") is the boundary of its correctness, not a hedge. The max differences above should be
  numerical noise (`~1e-7`, set by the diffuse-prior proxy `P0 = 1e8` in float64).
- **Two or more anchors:** the heuristic and the smoother diverge on both moments. The heuristic
  hard-switches to whichever anchor is nearer and freezes each anchor's disclosed value exactly; the
  smoother fuses every anchor's propagated estimate by inverse-variance weighting, which (a) blends
  the mean between anchors instead of jumping, (b) tightens the variance versus either one-sided
  propagation alone, and (c) revises the anchors themselves once more than one is disclosed.
- **The proposed method works and is exact here.** Treating each state prior as an independent
  univariate series, filtering forward (anchors = observations, nulls = predict-only) and smoothing
  backward is well-posed and, for this linear-Gaussian random walk, returns the *exact* posterior
  marginal. No simulation smoother is needed for this step.
- **When a simulation smoother would be needed instead:** (1) if the nonlinear positivity reparam
  $\lambda = e^{k a}$ is carried through the extension so the model is no longer linear-Gaussian, or
  (2) if we need *joint* posterior draws across time (correlated sample paths) rather than the
  per-step marginals the RTS pass returns. Neither applies to producing $(a_{\text{obs}},
  P_{\text{obs}})$ marginals, so RTS is the right tool here.
- **Practical read:** for a genuinely multi-anchor channel, `extend_states_prior_nearest`'s output is a
  conservative, discontinuous approximation -- correctly shaped, but neither as tight nor as smooth as
  a real smoother pass. That's consistent with the function's own framing: feed the extension into the
  augmented-measurement step as a prior, not as a final posterior. Wiring the in-house
  `kalman_filter_1d` + `kalman_rts_smoother_1d` in (as `extend_states_prior_smoothed` does here) is a candidate
  upgrade if multi-anchor channels are common in practice.